# Alleneremo un LLM per generare musica Jazz like

Definiamo un helper per processare l'audio

In [19]:
from music21 import converter, instrument, note, chord
import glob

def get_notes_from_midi(folder_path):
    notes = []

    # Cerchiamo tutti i file .mid nella cartella
    for file in glob.glob(f"{folder_path}/*.mid"):
        midi = converter.parse(file)
        print(f"Parsing: {file}")

        notes_to_parse = None

        # Proviamo a dividere per strumenti
        try:
            s2 = instrument.partitionByInstrument(midi)
            # Spesso nel jazz vogliamo solo il piano 
            notes_to_parse = s2.parts[0].recurse()
        except:
            # Se non ci sono parti definite, prendiamo tutto
            notes_to_parse = midi.flat.notes

        for element in notes_to_parse:
            # Se l'elemento è una nota singola
            if isinstance(element, note.Note):
                notes.append(str(element.pitch))
            elif isinstance(element, note.Rest):
                notes.append('rest')
            # Se l'elemento è un accordo (più note insieme)
            elif isinstance(element, chord.Chord):
                # Codifichiamo l'accordo come stringa di ID (es: 4.7.10)
                notes.append('.'.join(str(n) for n in element.normalOrder))

    return notes


all_notes = get_notes_from_midi("data/")

Parsing: data\2_of_a_kind_jp.mid
Parsing: data\500_miles_high-Chick-Corea_ee.mid
Parsing: data\99_miles_from_l_a-kar_rt.mid
Parsing: data\aint_that_a_kick_in_the_head_r2_rt.mid
Parsing: data\aint_we_got_fun_bz2-bz3.mid
Parsing: data\aja_sr3.mid
Parsing: data\alley_cat_bb10.mid
Parsing: data\all_blues-miles-davis_bl.mid
Parsing: data\all_blues-Miles-Davis_dz.mid
Parsing: data\all_my_tomorrows-kar-kos_mw.mid
Parsing: data\all_night_long_melod.mid
Parsing: data\all_of_me-1931-vs2-kar_jpp.mid
Parsing: data\all_of_you_mw.mid
Parsing: data\all_the_things_you_are-1940-Kern-Mayerl_jpp.mid
Parsing: data\all_the_things_you_are-2_dm.mid
Parsing: data\all_the_way-1957-vs2-kar_jpp.mid
Parsing: data\all_the_way_mw.mid
Parsing: data\almost_like_being_in_love_gw.mid
Parsing: data\alone_together1-MDavis_bl.mid
Parsing: data\alone_together2-MDavis_bl.mid
Parsing: data\always_and_forever_bnzo.mid
Parsing: data\amor_de_roca_cd.mid
Parsing: data\a_cottage_for_sale_rs.mid
Parsing: data\a_day_in_the_life_of_

In [27]:
# Otteniamo tutti i nomi delle note uniche
pitchnames = sorted(set(item for item in all_notes))
vocab_size = len(pitchnames)

# Creiamo un dizionario per mappare le note a numeri
note_to_int = {note: number for number, note in enumerate(pitchnames)}

# Prepariamo gli input (sequenze) e gli output (la nota successiva)
sequence_length = 100
network_input = []
network_output = []

for i in range(0, len(all_notes) - sequence_length):
    sequence_in = all_notes[i:i + sequence_length]
    sequence_out = all_notes[i + sequence_length]

    network_input.append([note_to_int[char] for char in sequence_in])
    network_output.append(note_to_int[sequence_out])

print(sequence_out)

rest


In [22]:
pitchnames

['0',
 '0.1',
 '0.1.2.6',
 '0.1.3.5.8',
 '0.1.4.7',
 '0.1.5.7',
 '0.1.5.8',
 '0.2',
 '0.2.3.5',
 '0.2.3.7',
 '0.2.4.6.9',
 '0.2.4.7',
 '0.2.4.7.9',
 '0.2.5',
 '0.2.6',
 '0.2.6.8',
 '0.2.7',
 '0.3',
 '0.3.5',
 '0.3.5.7.8',
 '0.3.5.8',
 '0.3.6.9',
 '0.3.7',
 '0.4',
 '0.4.6',
 '0.4.7',
 '0.4.8',
 '0.5',
 '0.5.6',
 '0.6',
 '1',
 '1.2.3.6.7',
 '1.3',
 '1.3.5.8',
 '1.3.7',
 '1.4',
 '1.4.6.8.9',
 '1.4.6.9',
 '1.4.7',
 '1.4.7.10',
 '1.4.8',
 '1.5',
 '1.5.7',
 '1.5.8',
 '1.6',
 '1.6.7',
 '1.7',
 '10',
 '10.0',
 '10.0.2.4',
 '10.0.2.4.7',
 '10.0.2.5',
 '10.0.2.5.7',
 '10.0.3.6',
 '10.0.5',
 '10.1',
 '10.1.3.6',
 '10.1.4',
 '10.1.5',
 '10.11',
 '10.11.0',
 '10.11.3.6',
 '10.2',
 '10.2.4',
 '10.2.4.5',
 '10.2.5',
 '10.3',
 '11',
 '11.0',
 '11.0.2.4.7',
 '11.0.4.5',
 '11.0.4.7',
 '11.1',
 '11.1.2.5.7',
 '11.1.3.5.7',
 '11.2',
 '11.2.4',
 '11.2.4.5',
 '11.2.4.5.7',
 '11.2.4.6.7',
 '11.2.4.7',
 '11.2.5',
 '11.2.5.7',
 '11.2.6',
 '11.3',
 '11.3.6',
 '11.4',
 '11.4.5',
 '2',
 '2.3',
 '2.3.5.7.10',
 '2.

In [24]:
import torch.nn as nn

class JazzLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers):
        super(JazzLSTM, self).__init__()
        # L'Embedding trasforma un numero (es. nota 42) in un vettore denso
        self.embedding = nn.Embedding(vocab_size, embed_dim) #one-hot e li trasforma in vettori più piccoli densi [0,0,0,1] [1.3, -2.4]

        # La LSTM vera e propria x = embed_dim 
        self.lstm = nn.LSTM(embed_dim, hidden_dim, n_layers,
                            batch_first=True, dropout=0.3)

        # Lo strato finale che decide qual è la prossima nota
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        # x shape: [batch_size, seq_len]
        x = self.embedding(x) # shape: [batch_size, seq_len, embed_dim]

        # out: tutti gli stati nascosti, _ : l'ultimo stato (a, c) SHAPE di out: [batch_size, seq_len, hidden_dim]
        out, _ = self.lstm(x)

        # Prendiamo solo l'output dell'ultimo step temporale
        out = self.fc(out[:, -1, :])
        return out

In [28]:
import torch
from torch.utils.data import Dataset, DataLoader

class MusicDataset(Dataset):
    def __init__(self, inputs, targets):
        # Convertiamo le liste in tensori di tipo Long (interi)
        self.inputs = torch.tensor(inputs, dtype=torch.long)
        self.targets = torch.tensor(targets, dtype=torch.long)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

# Parametri di addestramento
BATCH_SIZE = 64  # Quante sequenze vede la rete prima di aggiornare i pesi

# Inizializziamo Dataset e DataLoader
dataset = MusicDataset(network_input, network_output)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Test rapido: prendiamo un batch
data_iter = iter(train_loader)
sample_input, sample_target = next(data_iter)

print(f"Forma del batch input: {sample_input.shape}")   # [64, 100]
print(f"Forma del batch target: {sample_target.shape}") # [64]

Forma del batch input: torch.Size([64, 100])
Forma del batch target: torch.Size([64])


In [35]:
# Controlliamo se la GPU è disponibile
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Sto addestrando su: {device}")

# Iperparametri
VOCAB_SIZE = len(pitchnames)
EMBED_DIM = 256
HIDDEN_DIM = 512
N_LAYERS = 2
LR = 1e-4
# Inizializziamo il modello
model = JazzLSTM(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, N_LAYERS).to(device)



Sto addestrando su: cuda


In [36]:
# Perdita e Ottimizzatore
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [37]:
EPOCHS = 50 # Quante volte la rete vedrà l'intero dataset

model.train() # Mettiamo il modello in modalità addestramento
for epoch in range(EPOCHS):
    total_loss = 0

    for batch_idx, (sequences, targets) in enumerate(train_loader):
        # Spostiamo i dati sulla GPU/CPU
        sequences, targets = sequences.to(device), targets.to(device)

        # 1. Reset dei gradienti (fondamentale in PyTorch!)
        optimizer.zero_grad()

        # 2. Forward pass: la rete fa la sua previsione
        outputs = model(sequences)

        # 3. Calcolo dell'errore
        loss = criterion(outputs, targets)

        # 4. Backward pass: calcolo dei gradienti
        loss.backward()

        # 5. Ottimizzazione: aggiornamento dei pesi
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}")

Epoch [1/50], Loss: 3.2585
Epoch [2/50], Loss: 2.8664
Epoch [3/50], Loss: 2.5873
Epoch [4/50], Loss: 2.4015
Epoch [5/50], Loss: 2.2569
Epoch [6/50], Loss: 2.1421
Epoch [7/50], Loss: 2.0430
Epoch [8/50], Loss: 1.9428
Epoch [9/50], Loss: 1.8616


KeyboardInterrupt: 

In [30]:
import numpy as np
import torch.nn.functional as F

def generate_notes(model, network_input, pitchnames, n_generate=500, temperature=1.0):
    model.eval() # Modalità valutazione

    # Mapping inverso: da numero a nota
    int_to_note = {number: note for number, note in enumerate(pitchnames)}

    # Scegliamo una sequenza casuale come seme iniziale
    start = np.random.randint(0, len(network_input)-1)
    pattern = network_input[start]
    prediction_output = []

    # Generiamo n note
    with torch.no_grad():
        for note_index in range(n_generate):
            prediction_input = torch.tensor([pattern], dtype=torch.long).to(device)

            # Forward pass
            output = model(prediction_input)

            # Applichiamo la Temperatura
            # Più T è alta, più la distribuzione diventa "piatta" (caotica)
            output = output / temperature
            prob_dist = F.softmax(output, dim=1).cpu().numpy().reshape(-1)

            # Scegliamo la prossima nota in base alla distribuzione di probabilità
            predicted_index = np.random.choice(len(pitchnames), p=prob_dist)

            # Convertiamo l'indice in nota e salviamo
            result = int_to_note[predicted_index]
            prediction_output.append(result)

            # Aggiorniamo il pattern: togliamo la prima nota e aggiungiamo la nuova
            pattern.append(predicted_index)
            pattern = pattern[1:]

    return prediction_output

In [31]:
from music21 import stream, note, chord, instrument

def create_midi(prediction_output, filename='jazz_output.mid'):
    offset = 0
    output_notes = []

    for pattern in prediction_output:
        # 1. GESTIONE DELLE PAUSE
        if pattern == 'R' or pattern == 'rest':
            new_element = note.Rest()
            new_element.offset = offset
            output_notes.append(new_element)

        # 2. GESTIONE DEGLI ACCORDI (es: '0.4.7' o '10')
        elif ('.' in pattern) or pattern.isdigit():
            notes_in_chord = pattern.split('.')
            notes = []
            for current_note in notes_in_chord:
                new_note = note.Note(int(current_note))
                #new_note.storedInstrument = instrument.BaritoneSaxophone()
                notes.append(new_note)
            new_element = chord.Chord(notes)
            new_element.offset = offset
            output_notes.append(new_element)

        # 3. GESTIONE DELLE NOTE SINGOLE (es: 'C4')
        else:
            new_element = note.Note(pattern)
            new_element.offset = offset
            #new_element.storedInstrument = instrument.BaritoneSaxophone()
            output_notes.append(new_element)

        # Aumentiamo l'offset (la posizione nel tempo)
        # 0.5 equivale a una croma (ottavo) in un tempo standard
        offset += 0.5

    midi_stream = stream.Stream(output_notes)
    midi_stream.write('midi', fp=filename)
    print(f"File salvato con successo: {filename}")

In [32]:
# 1. Genera la sequenza (usa T=0.8 per un jazz "stabile" o T=1.2 per "improvvisazione spinta")
generated_notes = generate_notes(model, network_input, pitchnames, temperature=3)
print(generated_notes[:100])
# 2. Crea il file MIDI


['A1', 'E-2', 'G#1', 'A4', '3.5.8.11', 'D2', 'B2', 'C#3', 'rest', 'E-3', 'D5', 'D3', '0.4.6', 'rest', 'E-3', '9.10.0.2.5', 'E-2', 'D2', 'B-5', 'rest', 'C4', '4.5', 'F#4', 'rest', 'G2', '11.0', 'rest', 'A2', 'B-1', '11.0', 'rest', '0.1.3.5.8', 'rest', 'F#4', 'D2', '0.2', '0.2', 'rest', 'B2', 'G#3', 'C#2', 'rest', 'E-5', 'B1', 'B2', '3.5.8', 'rest', '1.3.7', 'B2', 'C#2', 'G#4', 'A4', 'G#2', 'D2', 'B1', '9.1.2', 'rest', 'D1', 'E6', 'D2', 'A1', 'rest', '0.2', '4.5.9.11', 'E3', '4.8.11', 'rest', 'rest', 'C3', '5.8', 'B1', '4.7.9.0', 'rest', 'G1', '5.9', 'A6', '3.5.8.11', 'C2', '11.2.4.7', 'G5', 'F1', '3', 'A5', 'E-2', 'E5', '8.11', '0.3.5.7.8', 'rest', 'F6', 'A4', '6.11', 'G3', 'C4', '1.4.7', 'F5', 'F2', '8.11', 'rest', 'B5', 'D4']


In [33]:
create_midi(generated_notes)

File salvato con successo: jazz_output.mid


In [39]:
torch.load(model,'model.pb')